In [6]:
pip install pandas-datareader pandas

     -------------------------------------- 109.5/109.5 kB 3.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.trend import MACD, PSARIndicator
from ta.volatility import BollingerBands
from ta.utils import dropna

In [3]:
import datetime
import pandas as pd
import pandas_datareader.data as web

# 1. Set the exact historical date range boundaries
start_date = datetime.datetime(2000, 1, 1)
end_date = datetime.datetime(2026, 1, 1)

try:
    print("Opening secure API bridge to FRED...")
    
    # 2. Extract the official EUR/USD exchange spot rates ('DEXUSEU')
    df = web.DataReader("DEXUSEU", "fred", start_date, end_date)
    
    # 3. Clean up the resulting data structure
    df.columns = ["Close"]
    df.index.name = "Date"
    
    # Drop weekend closures or banking holidays where rates aren't posted
    df = df.dropna()
    
    print("\n--- Data Download Successful! ---")
    print(df.head())
    print(f"\nTotal historical trading sessions loaded: {len(df)}")
    
    # 4. Save a clean copy locally to ensure offline persistence
    df.to_csv("EURUSD_Historical_Data.csv")
    print("Dataset safely written to 'EURUSD_Historical_Data.csv'")

except Exception as e:
    print(f"\nConnection aborted: {e}")
    print("If this fails, a local network firewall rule is dropping outbound terminal connections.")


Opening secure API bridge to FRED...

--- Data Download Successful! ---
             Close
Date              
2000-01-03  1.0155
2000-01-04  1.0309
2000-01-05  1.0335
2000-01-06  1.0324
2000-01-07  1.0294

Total historical trading sessions loaded: 6518
Dataset safely written to 'EURUSD_Historical_Data.csv'


In [4]:
df = pd.read_csv('EURUSD_Historical_Data.csv')
print(df.shape)
df.head()

(6518, 2)


,Date,Close
0,2000-01-03,1.0155
1,2000-01-04,1.0309
2,2000-01-05,1.0335
3,2000-01-06,1.0324
4,2000-01-07,1.0294


# EUR/USD 

In [8]:
import yfinance as yf

ticker_symbol = "EURUSD=X"

try:
    print(f"Connecting to financial servers for {ticker_symbol}...")
    
    # 1. Download authentic historical OHLCV data using the core engine
    # Setting auto_adjust=False preserves the separate 'Close' and 'Adj Close' columns
    df = yf.download(ticker_symbol, start="2000-01-01", end="2026-01-01", auto_adjust=False)
    
    if not df.empty:
        # 2. Clean the modern MultiIndex column structure to match your requirements
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel('Ticker')
            
        # 3. Sort chronologically from oldest to newest
        df = df.sort_index()
        
        # 4. Filter down to the exact 6 columns you requested
        final_columns = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
        df_final = df[final_columns]
        
        print("\n--- Authentic Historical Data Loaded Successfully! ---")
        print(df_final.head())
        print(f"\nTotal trading sessions captured: {len(df_final)}")
        
        # Save a backup copy locally to your computer
        df_final.to_csv("EURUSD_Official_OHLCV.csv")
        print("Dataset safely saved completely offline to 'EURUSD_Official_OHLCV.csv'")
    else:
        print("The data download returned empty. Please double-check your package installation.")

except Exception as e:
    print(f"\nAn error occurred during extraction: {e}")


Connecting to financial servers for EURUSD=X...


[*********************100%***********************]  1 of 1 completed



--- Authentic Historical Data Loaded Successfully! ---
Price           Open      High       Low     Close  Adj Close  Volume
Date                                                                 
2003-12-01  1.203398  1.204007  1.194401  1.196501   1.196501       0
2003-12-02  1.196101  1.210903  1.194600  1.208897   1.208897       0
2003-12-03  1.209000  1.213003  1.207700  1.212298   1.212298       0
2003-12-04  1.212004  1.214403  1.204398  1.208094   1.208094       0
2003-12-05  1.207802  1.219096  1.206593  1.218695   1.218695       0

Total trading sessions captured: 5730
Dataset safely saved completely offline to 'EURUSD_Official_OHLCV.csv'


In [3]:
df = pd.read_csv('EURUSD_Official_OHLCV.csv')
print(df.shape)
df.head()

(5730, 7)


,Date,Open,High,Low,Close,Adj Close,Volume
0,2003-12-01,1.203398,1.204007,1.194401,1.196501,1.196501,0
1,2003-12-02,1.196101,1.210903,1.194600,1.208897,1.208897,0
2,2003-12-03,1.209000,1.213003,1.207700,1.212298,1.212298,0
3,2003-12-04,1.212004,1.214403,1.204398,1.208094,1.208094,0
4,2003-12-05,1.207802,1.219096,1.206593,1.218695,1.218695,0
